### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [ ]:
# Import necessary libraries for AutoGen distributed runtime
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

# Configuration flag: True = all agents on single worker, False = distributed across multiple workers
# This demonstrates the difference between centralized vs distributed agent deployment
ALL_IN_ONE_WORKER = False

### Start with our Message class

In [ ]:
# Simple message class for agent communication in distributed environment
# In distributed systems, messages need to be serializable to pass between processes
@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [ ]:
# GRPC Host: Central coordination point for distributed AutoGen agents
# This host manages communication between distributed workers using GRPC protocol
# Think of it as the "traffic controller" that routes messages between agents on different machines
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [ ]:
# Set up web search tool using Google Serper API
# This tool will be used by agents to research information about AutoGen
# LangChainToolAdapter wraps the LangChain tool for use in AutoGen distributed environment
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [ ]:
# Define research tasks for distributed agent collaboration
# This demonstrates how different agents can work on complementary tasks

# Task 1: Research pros of AutoGen (assigned to Player1Agent)
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

# Task 2: Research cons of AutoGen (assigned to Player2Agent)  
instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

# Final decision task: Synthesize research from both agents (handled by Judge)
judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [ ]:
# Define distributed agents that can run on separate workers/machines
# Each agent is designed to be stateless and communicates via messages

class Player1Agent(RoutedAgent):
    """Agent specialized in researching pros/advantages of AutoGen"""
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        # Include search tool for research capability and reflection for better responses
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    """Agent specialized in researching cons/disadvantages of AutoGen"""
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        # Same capabilities as Player1 but will receive different research tasks
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    """Orchestrator agent that coordinates research and makes final decision"""
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        # Judge doesn't need search tools - it synthesizes research from other agents
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        # Distribute research tasks to specialized agents
        message1 = Message(content=instruction1)  # Pros research
        message2 = Message(content=instruction2)  # Cons research
        
        # Send messages to distributed agents (could be on different machines)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        
        # Await responses from distributed agents
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        
        # Compile research results
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        
        # Make final decision based on compiled research
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)

In [ ]:
# Configure distributed or centralized runtime based on ALL_IN_ONE_WORKER flag
# This demonstrates two deployment patterns for AutoGen agents

from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    # CENTRALIZED DEPLOYMENT: All agents run on a single worker process
    # Simpler setup, good for development and smaller workloads
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    # Register all agents on the same worker
    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:
    # DISTRIBUTED DEPLOYMENT: Each agent runs on its own worker process
    # Better scalability, fault tolerance, and resource isolation
    # In production, these could run on completely separate machines
    
    # Worker 1: Handles Player1Agent (pros research)
    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    # Worker 2: Handles Player2Agent (cons research)  
    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    # Worker 3: Handles Judge (orchestration and decision)
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")

In [ ]:
# Trigger the distributed research workflow
# The Judge will coordinate with Player1 and Player2 agents to gather research
# then make a final decision about using AutoGen
response = await worker.send_message(Message(content="Go!"), agent_id)

In [ ]:
# Display the comprehensive research report and decision
# Shows pros, cons, and final recommendation from the distributed agent team
display(Markdown(response.content))

In [ ]:
# Clean shutdown of distributed workers
# Important: Stop all workers to release resources and close connections
await worker.stop()
if not ALL_IN_ONE_WORKER:
    # In distributed mode, stop all individual workers
    await worker1.stop()
    await worker2.stop()

In [ ]:
# Stop the GRPC host server
# This shuts down the central coordination point for the distributed system
await host.stop()